In [2]:
import torch, torch.nn as nn, torchvision

In [3]:
from torchvision.models import resnet101, ResNet101_Weights
model = resnet101(weights=ResNet101_Weights.DEFAULT)

In [4]:
num_iterations = 1
xe = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)

# Basic example

In [5]:
desired_batch_size = 100
tolerated_batch_size = 20
accum_steps = desired_batch_size//tolerated_batch_size

for i in range(num_iterations):
    inputs = torch.randn(tolerated_batch_size, 3, 224, 224)
    labels = torch.randint(0, 100, (tolerated_batch_size,))
    loss = xe(model(inputs), labels)
    loss /= accum_steps
    loss.backward()

    if (i+1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

    print(f"Done with batch {i+1}")

Done with batch 1


# CNN with gradient accumulation

In [5]:
train_loader = DataLoader(dataset, batch_size=tolerated_batch_size, shuffle=True)
accum_steps = desired_batch_size // tolerated_batch_size  # ví dụ 100 // 20 = 5
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
xe = nn.CrossEntropyLoss()

model.train()
for i, (imgs, labels) in enumerate(train_loader):

    outputs = model(imgs)
    loss = xe(outputs, labels)

    loss = loss / accum_steps
    loss.backward()  # accumulate gradient

    if (i + 1) % accum_steps == 0:
        optimizer.step()
        optimizer.zero_grad()  # reset gradient

if len(train_loader) % accum_steps != 0:
    optimizer.step()
    optimizer.zero_grad()